# Text Preprocessing Exercise (Using Basic Python)

**Student:** Harutyun Abajyan
**Student ID:** 36111

In this notebook I preprocess the given text using **only built-in Python** (string methods, lists, sets and
dictionaries). There is no `import` anywhere and no NLTK, so every step is written by hand.

Steps:
1. Tokenization
2. Lowercasing
3. Punctuation removal
4. Stop word removal
5. Stemming (simple suffix removal)

Bonus: a small dictionary-based lemmatizer, compared with my stemmer.

In [1]:
# The text from the task
text = ("Natural Language Processing (NLP) is a subfield of linguistics, computer science, "
        "and artificial intelligence concerned with the interactions between computers and "
        "human language. It's used to analyze text, allowing machines to understand, "
        "interpret, and manipulate human language. NLP has many real-world applications, "
        "including machine translation, sentiment analysis, and chatbots.")
print(text)

Natural Language Processing (NLP) is a subfield of linguistics, computer science, and artificial intelligence concerned with the interactions between computers and human language. It's used to analyze text, allowing machines to understand, interpret, and manipulate human language. NLP has many real-world applications, including machine translation, sentiment analysis, and chatbots.


## Step 1: Tokenization

The easiest way to split a text into words is `str.split()`. Without an argument it splits on any
whitespace (spaces, tabs, new lines) and does not create empty tokens when there are two spaces in a row.

Punctuation is still attached to the words after this step (for example `"(NLP)"` or `"linguistics,"`).
That is fine, it is removed in step 3.

In [2]:
def tokenize(text):
    # split() without an argument cuts the text at every block of whitespace
    return text.split()

tokens = tokenize(text)
print("Number of tokens:", len(tokens))
print(tokens)

Number of tokens: 49
['Natural', 'Language', 'Processing', '(NLP)', 'is', 'a', 'subfield', 'of', 'linguistics,', 'computer', 'science,', 'and', 'artificial', 'intelligence', 'concerned', 'with', 'the', 'interactions', 'between', 'computers', 'and', 'human', 'language.', "It's", 'used', 'to', 'analyze', 'text,', 'allowing', 'machines', 'to', 'understand,', 'interpret,', 'and', 'manipulate', 'human', 'language.', 'NLP', 'has', 'many', 'real-world', 'applications,', 'including', 'machine', 'translation,', 'sentiment', 'analysis,', 'and', 'chatbots.']


## Steps 2 and 3: Lowercasing and punctuation removal

**Lowercasing:** "NLP" and "nlp", or "Language" and "language", should count as the same word, so I call
`str.lower()` on every token.

**Punctuation removal:** I wrote all punctuation characters into one string (the same characters as
`string.punctuation`, plus curly quotes because they often appear in copied text). For every token I keep only
the characters that are not in this string. If a token is only punctuation (for example a single `"-"`), it
becomes empty and I drop it.

Note: this also removes punctuation *inside* a word, so `"it's"` becomes `"its"` and `"real-world"` becomes
`"realworld"`. For this exercise that is acceptable; a better tokenizer would split such words first.

In [3]:
def to_lowercase(tokens):
    return [token.lower() for token in tokens]


# all ASCII punctuation characters + curly quotes
PUNCTUATION = "!\"#$%&'()*+,-./:;<=>?@[\\]^_`{|}~" + "‘’“”"

def remove_punctuation(tokens):
    cleaned = []
    for token in tokens:
        # keep only the characters that are not punctuation
        new_token = "".join(ch for ch in token if ch not in PUNCTUATION)
        if new_token != "":      # a token made only of punctuation disappears
            cleaned.append(new_token)
    return cleaned


lower_tokens = to_lowercase(tokens)            # step 2
clean_tokens = remove_punctuation(lower_tokens)  # step 3
print("Number of tokens:", len(clean_tokens))
print(clean_tokens)

Number of tokens: 49
['natural', 'language', 'processing', 'nlp', 'is', 'a', 'subfield', 'of', 'linguistics', 'computer', 'science', 'and', 'artificial', 'intelligence', 'concerned', 'with', 'the', 'interactions', 'between', 'computers', 'and', 'human', 'language', 'its', 'used', 'to', 'analyze', 'text', 'allowing', 'machines', 'to', 'understand', 'interpret', 'and', 'manipulate', 'human', 'language', 'nlp', 'has', 'many', 'realworld', 'applications', 'including', 'machine', 'translation', 'sentiment', 'analysis', 'and', 'chatbots']


In [4]:
# Which tokens were changed by step 3?
for token in lower_tokens:
    without_punct = "".join(ch for ch in token if ch not in PUNCTUATION)
    if without_punct != token:
        print(f"{token:16} -> {without_punct}")

(nlp)            -> nlp
linguistics,     -> linguistics
science,         -> science
language.        -> language
it's             -> its
text,            -> text
understand,      -> understand
interpret,       -> interpret
language.        -> language
real-world       -> realworld
applications,    -> applications
translation,     -> translation
analysis,        -> analysis
chatbots.        -> chatbots


## Step 4: Stop word removal

I start with the list from the hint and add a few more words that appear in this text and do not carry
meaning: `"it"`, `"its"` (comes from "It's"), `"with"`, `"between"` and `"has"`.
The stop words are stored in a **set**, because checking `word in set` is faster than searching a list.

In [5]:
STOP_WORDS = {"the", "a", "an", "in", "on", "at", "for", "to", "of", "and", "is", "are",  # list from the hint
              "it", "its", "with", "between", "has"}                                       # my additions

def remove_stop_words(tokens):
    return [token for token in tokens if token not in STOP_WORDS]


no_stop_tokens = remove_stop_words(clean_tokens)
removed_words = [token for token in clean_tokens if token in STOP_WORDS]

print("Number of tokens:", len(no_stop_tokens))
print(no_stop_tokens)
print()
print("Removed", len(removed_words), "stop words:", removed_words)

Number of tokens: 35
['natural', 'language', 'processing', 'nlp', 'subfield', 'linguistics', 'computer', 'science', 'artificial', 'intelligence', 'concerned', 'interactions', 'computers', 'human', 'language', 'used', 'analyze', 'text', 'allowing', 'machines', 'understand', 'interpret', 'manipulate', 'human', 'language', 'nlp', 'many', 'realworld', 'applications', 'including', 'machine', 'translation', 'sentiment', 'analysis', 'chatbots']

Removed 14 stop words: ['is', 'a', 'of', 'and', 'with', 'the', 'between', 'and', 'its', 'to', 'to', 'and', 'has', 'and']


## Step 5: Stemming

My stemmer is a list of `(suffix, replacement)` rules. The rules are checked from top to bottom and the
**first** rule whose suffix matches the end of the word is used. Longer suffixes come before shorter ones
(`"ions"` before `"ion"`, `"ers"` before `"er"`), otherwise the shorter rule would always win.

Two safety rules:
* the stem must keep at least 3 letters, otherwise the word is not changed (so `"used"` does not become `"us"`
  and `"sing"` does not become `"s"`);
* some rules replace a suffix with itself (`"ss" -> "ss"`). They do nothing, but because they match first they
  protect words like `"class"` or `"analysis"` from losing their last `s`.

It is a very simplified version of the first steps of the Porter stemmer.

In [6]:
SUFFIX_RULES = [
    ("sses", "ss"),  # classes      -> class
    ("ies",  "y"),   # studies      -> study
    ("ss",   "ss"),  # class        -> class     (protection: keep the last s)
    ("is",   "is"),  # analysis     -> analysis  (protection)
    ("us",   "us"),  # status       -> status    (protection)
    ("ing",  ""),    # processing   -> process
    ("ions", ""),    # interactions -> interact
    ("ion",  ""),    # translation  -> translat
    ("ers",  ""),    # computers    -> comput
    ("er",   ""),    # computer     -> comput
    ("ed",   ""),    # concerned    -> concern
    ("ly",   ""),    # quickly      -> quick
    ("s",    ""),    # chatbots     -> chatbot
]
MIN_STEM_LENGTH = 3

def simple_stem(word):
    for suffix, replacement in SUFFIX_RULES:
        if word.endswith(suffix):
            stem = word[:len(word) - len(suffix)] + replacement
            if len(stem) >= MIN_STEM_LENGTH:
                return stem
            return word      # the stem would be too short -> keep the word unchanged
    return word              # no rule matched

def stem_tokens(tokens):
    return [simple_stem(token) for token in tokens]


stemmed_tokens = stem_tokens(no_stop_tokens)
print("Number of tokens:", len(stemmed_tokens))
print(stemmed_tokens)

Number of tokens: 35
['natural', 'language', 'process', 'nlp', 'subfield', 'linguistic', 'comput', 'science', 'artificial', 'intelligence', 'concern', 'interact', 'comput', 'human', 'language', 'used', 'analyze', 'text', 'allow', 'machine', 'understand', 'interpret', 'manipulate', 'human', 'language', 'nlp', 'many', 'realworld', 'applicat', 'includ', 'machine', 'translat', 'sentiment', 'analysis', 'chatbot']


In [7]:
# Which words did the stemmer change? (each word once, in the order of the text)
for word in dict.fromkeys(no_stop_tokens):
    if simple_stem(word) != word:
        print(f"{word:14} -> {simple_stem(word)}")

processing     -> process
linguistics    -> linguistic
computer       -> comput
concerned      -> concern
interactions   -> interact
computers      -> comput
allowing       -> allow
machines       -> machine
applications   -> applicat
including      -> includ
translation    -> translat
chatbots       -> chatbot


## Expected output (all four lists together)

In [8]:
print("1. Tokens after step 1 (tokenization):")
print(tokens, "\n")
print("2. Tokens after steps 2 and 3 (lowercase + no punctuation):")
print(clean_tokens, "\n")
print("3. Tokens after step 4 (stop word removal):")
print(no_stop_tokens, "\n")
print("4. Final tokens after step 5 (simple stemming):")
print(stemmed_tokens, "\n")

print("Number of tokens per step:", len(tokens), "->", len(clean_tokens), "->",
      len(no_stop_tokens), "->", len(stemmed_tokens))
print("Different words per step: ", len(set(tokens)), "->", len(set(clean_tokens)), "->",
      len(set(no_stop_tokens)), "->", len(set(stemmed_tokens)))

1. Tokens after step 1 (tokenization):
['Natural', 'Language', 'Processing', '(NLP)', 'is', 'a', 'subfield', 'of', 'linguistics,', 'computer', 'science,', 'and', 'artificial', 'intelligence', 'concerned', 'with', 'the', 'interactions', 'between', 'computers', 'and', 'human', 'language.', "It's", 'used', 'to', 'analyze', 'text,', 'allowing', 'machines', 'to', 'understand,', 'interpret,', 'and', 'manipulate', 'human', 'language.', 'NLP', 'has', 'many', 'real-world', 'applications,', 'including', 'machine', 'translation,', 'sentiment', 'analysis,', 'and', 'chatbots.'] 

2. Tokens after steps 2 and 3 (lowercase + no punctuation):
['natural', 'language', 'processing', 'nlp', 'is', 'a', 'subfield', 'of', 'linguistics', 'computer', 'science', 'and', 'artificial', 'intelligence', 'concerned', 'with', 'the', 'interactions', 'between', 'computers', 'and', 'human', 'language', 'its', 'used', 'to', 'analyze', 'text', 'allowing', 'machines', 'to', 'understand', 'interpret', 'and', 'manipulate', 'hu

## Bonus: Lemmatization with a dictionary

A lemmatizer returns the dictionary form (lemma) of a word. Real lemmatizers (like WordNet in NLTK) use a big
dictionary and the part of speech. My version is just a Python `dict`: if the word is in it, I return its base
form, otherwise I return the word unchanged.

The dictionary has the inflected words of this text and some common irregular forms (to show what a stemmer
cannot do). `"processing"` is not in it on purpose: in "Natural Language Processing" it is a noun, and its lemma
is `"processing"` itself.

In [9]:
LEMMA_DICT = {
    # forms of "be" and "have"
    "is": "be", "are": "be", "was": "be", "were": "be", "been": "be", "am": "be",
    "has": "have", "had": "have", "having": "have",
    # plural nouns from the text
    "computers": "computer", "machines": "machine", "interactions": "interaction",
    "applications": "application", "chatbots": "chatbot",
    # verb forms from the text
    "used": "use", "allowing": "allow", "including": "include", "concerned": "concern",
    # some irregular words
    "went": "go", "gone": "go", "children": "child", "mice": "mouse", "better": "good",
    "studies": "study", "running": "run", "ran": "run",
}

def lemmatize(word):
    return LEMMA_DICT.get(word, word)

def lemmatize_tokens(tokens):
    return [lemmatize(token) for token in tokens]


lemmatized_tokens = lemmatize_tokens(no_stop_tokens)
print("Number of tokens:", len(lemmatized_tokens))
print(lemmatized_tokens)

Number of tokens: 35
['natural', 'language', 'processing', 'nlp', 'subfield', 'linguistics', 'computer', 'science', 'artificial', 'intelligence', 'concern', 'interaction', 'computer', 'human', 'language', 'use', 'analyze', 'text', 'allow', 'machine', 'understand', 'interpret', 'manipulate', 'human', 'language', 'nlp', 'many', 'realworld', 'application', 'include', 'machine', 'translation', 'sentiment', 'analysis', 'chatbot']


In [10]:
# Side-by-side comparison on the same input (the tokens after step 4)
print(f"{'word':14} {'stem':14} {'lemma':14}")
print("-" * 42)
for word in dict.fromkeys(no_stop_tokens):
    stem, lemma = simple_stem(word), lemmatize(word)
    mark = "" if stem == lemma else "   <- different"
    print(f"{word:14} {stem:14} {lemma:14}{mark}")

word           stem           lemma         
------------------------------------------
natural        natural        natural       
language       language       language      
processing     process        processing       <- different
nlp            nlp            nlp           
subfield       subfield       subfield      
linguistics    linguistic     linguistics      <- different
computer       comput         computer         <- different
science        science        science       
artificial     artificial     artificial    
intelligence   intelligence   intelligence  
concerned      concern        concern       
interactions   interact       interaction      <- different
computers      comput         computer         <- different
human          human          human         
used           used           use              <- different
analyze        analyze        analyze       
text           text           text          
allowing       allow          allow         
machines    

In [11]:
# A few extra words that are not in the text, to show the difference better
extra_words = ["is", "are", "has", "went", "children", "better", "studies", "running", "classes", "happily"]
print(f"{'word':14} {'stem':14} {'lemma':14}")
print("-" * 42)
for word in extra_words:
    print(f"{word:14} {simple_stem(word):14} {lemmatize(word):14}")

word           stem           lemma         
------------------------------------------
is             is             be            
are            are            be            
has            has            have          
went           went           go            
children       children       child         
better         bett           good          
studies        study          study         
running        runn           run           
classes        class          classes       
happily        happi          happily       


### Comparison of stemming and lemmatization

* **Stems are often not real words:** `comput`, `translat`, `applicat`, `includ`. The lemmatizer always gives
  real words: `computer`, `translation`, `application`, `include`.
* **Both of them group word forms together:** "computer" and "computers" become `comput` with the stemmer and
  `computer` with the lemmatizer, so in both cases they are counted as one word.
* **The stemmer can make mistakes:** `linguistics` (the science) becomes `linguistic` (an adjective), and
  `used` is not changed at all because the stem `us` would be too short. The lemmatizer gives `use`.
  In the extra words, `better` becomes `bett` (the `er` rule thinks it is like "computer") and `running`
  becomes `runn` (my rules do not remove the double consonant, the real Porter stemmer does).
* **Irregular forms:** the stemmer cannot know that "is"/"are" come from "be" or "went" from "go", because
  there is no common suffix. The lemmatizer handles them because they are in the dictionary. On the other side,
  `happily` gives `happi` with the stemmer and stays unchanged with the lemmatizer, because it is not in my
  dictionary.
* **Coverage:** the stemmer works on any word, also words it has never seen. My lemmatizer only knows the words I
  put in the dictionary, so for real texts it would need a very large dictionary (this is why NLTK uses WordNet).

## Summary

I used only built-in Python: `split()` for tokenization, `lower()` for lowercasing, a character filter for
punctuation, a set of stop words, and a list of suffix rules for stemming. The text has 49 tokens. Punctuation
removal did not delete any token (there is no token made only of punctuation), but it changed 14 of them, for
example `"(nlp)"` to `"nlp"`. Stop word removal deleted 14 tokens, leaving 35. Stemming does not change the
number of tokens; it changed 12 different words and reduced the number of different words from 31 to 29,
because `computer`/`computers` both became `comput` and `machine`/`machines` both became `machine`.